# 01E — Lead Features Incremental Lab
## bd_replica_crm · optimización real para LIVE

Objetivo: comparar la lógica actual de features con una estrategia incremental basada en preagregaciones diarias por proyecto, asesor y global.

Contexto confirmado:
- 205,947 evidencias pendientes históricas.
- 1,930 pendientes LIVE.
- `LATERAL` fue 100% equivalente, pero 0.81x: más lento.

Este notebook es read-only por defecto.


In [1]:
from __future__ import annotations
import sys, time
from pathlib import Path
import pandas as pd
import numpy as np

cwd=Path.cwd().resolve()
PROJECT_ROOT=cwd if (cwd/"pyproject.toml").exists() else cwd.parent
if not (PROJECT_ROOT/"pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")

SRC=PROJECT_ROOT/"src"
if str(SRC) not in sys.path:
    sys.path.insert(0,str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres
from replica_cygnus.lead_scoring.config import load_lead_scoring_config

settings=load_settings(PROJECT_ROOT)
config_path=PROJECT_ROOT/"config"/"lead_scoring.yml"
if not config_path.exists():
    config_path=PROJECT_ROOT/"config"/"lead_scoring.example.yml"
cfg=load_lead_scoring_config(config_path)
conn=connect_postgres(settings)

pd.set_option("display.max_columns",160)
pd.set_option("display.max_rows",160)
pd.set_option("display.width",240)

def df(sql,params=None):
    return pd.read_sql_query(sql,conn,params=params)

SEP_H=int(cfg.sep_horizon_days)
MINUTA_H=int(cfg.minuta_horizon_days)
SCORE_WINDOW=int(cfg.score_window_days)

print("DB:",settings.postgres.database)
print("score_window_days:",SCORE_WINDOW)


DB: medallio_dw
score_window_days: 14


## 1. Estado actual


In [2]:
status=df(f"""
SELECT
 COUNT(*) AS total,
 COUNT(*) FILTER (WHERE features_refreshed_at IS NULL) AS pending,
 COUNT(*) FILTER (
   WHERE features_refreshed_at IS NULL
     AND decision_at>=current_date-({SCORE_WINDOW}*interval '1 day')
 ) AS pending_live
FROM features.lead_evidence
""")
status.T


C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


,0
total,206029
pending,206029
pending_live,2012


## 2. Índices actuales


In [3]:
indexes=df("""
SELECT indexname,indexdef
FROM pg_indexes
WHERE schemaname='features' AND tablename='lead_evidence'
ORDER BY indexname
""")
indexes


C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


,indexname,indexdef
0,ix_lead_evidence_advisor_time,CREATE INDEX ix_lead_evidence_advisor_time ON ...
1,ix_lead_evidence_decision_at,CREATE INDEX ix_lead_evidence_decision_at ON f...
2,ix_lead_evidence_document_time,CREATE INDEX ix_lead_evidence_document_time ON...
3,ix_lead_evidence_project_time,CREATE INDEX ix_lead_evidence_project_time ON ...
4,lead_evidence_pkey,CREATE UNIQUE INDEX lead_evidence_pkey ON feat...


## 3. Baseline actual


In [4]:
def baseline_query(limit_n=100):
    return f"""
WITH target AS (
 SELECT *
 FROM features.lead_evidence
 WHERE features_refreshed_at IS NULL
   AND decision_at>=current_date-({SCORE_WINDOW}*interval '1 day')
 ORDER BY decision_at,evidence_key
 LIMIT {int(limit_n)}
)
SELECT
 e.evidence_key,e.decision_at,e.documento_cliente,e.codigo_proyecto,e.asesor,
 COALESCE((SELECT COUNT(*) FROM features.lead_evidence p
   WHERE p.documento_cliente=e.documento_cliente
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'),0) AS client_prior_assignments_90d,
 (SELECT EXTRACT(EPOCH FROM (e.decision_at-MAX(p.decision_at)))/86400.0
   FROM features.lead_evidence p
   WHERE p.documento_cliente=e.documento_cliente
     AND p.decision_at<e.decision_at) AS days_since_previous_assignment,
 COALESCE((SELECT COUNT(*) FROM features.lead_evidence p
   WHERE p.codigo_proyecto=e.codigo_proyecto
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'),0) AS project_leads_90d,
 (SELECT AVG(p.separacion_14d::double precision)
   FROM features.lead_evidence p
   WHERE p.codigo_proyecto=e.codigo_proyecto
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'
     AND p.separacion_14d IS NOT NULL
     AND p.decision_at+interval '{SEP_H} days'<=e.decision_at) AS project_sep_rate_90d,
 (SELECT AVG(p.minuta_60d::double precision)
   FROM features.lead_evidence p
   WHERE p.codigo_proyecto=e.codigo_proyecto
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '180 days'
     AND p.minuta_60d IS NOT NULL
     AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at) AS project_minuta_rate_180d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE COALESCE((SELECT COUNT(*)
   FROM features.lead_evidence p
   WHERE p.asesor=e.asesor
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'),0) END AS advisor_leads_90d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE (SELECT AVG(p.separacion_14d::double precision)
   FROM features.lead_evidence p
   WHERE p.asesor=e.asesor
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'
     AND p.separacion_14d IS NOT NULL
     AND p.decision_at+interval '{SEP_H} days'<=e.decision_at) END AS advisor_sep_rate_90d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE (SELECT AVG(p.minuta_60d::double precision)
   FROM features.lead_evidence p
   WHERE p.asesor=e.asesor
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '180 days'
     AND p.minuta_60d IS NOT NULL
     AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at) END AS advisor_minuta_rate_180d,
 (SELECT AVG(p.separacion_14d::double precision)
   FROM features.lead_evidence p
   WHERE p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'
     AND p.separacion_14d IS NOT NULL
     AND p.decision_at+interval '{SEP_H} days'<=e.decision_at) AS global_sep_rate_90d,
 (SELECT AVG(p.minuta_60d::double precision)
   FROM features.lead_evidence p
   WHERE p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '180 days'
     AND p.minuta_60d IS NOT NULL
     AND p.decision_at+interval '{MINUTA_H} days'<=e.decision_at) AS global_minuta_rate_180d
FROM target e
ORDER BY e.decision_at,e.evidence_key
"""


## 4. Incremental con preagregación diaria


In [5]:
def incremental_query(limit_n=100):
    return f"""
WITH target AS (
 SELECT *
 FROM features.lead_evidence
 WHERE features_refreshed_at IS NULL
   AND decision_at>=current_date-({SCORE_WINDOW}*interval '1 day')
 ORDER BY decision_at,evidence_key
 LIMIT {int(limit_n)}
),
project_daily AS (
 SELECT codigo_proyecto,decision_at::date d,
        COUNT(*) leads_day,
        SUM(separacion_14d) FILTER (WHERE separacion_14d IS NOT NULL) sep_pos_day,
        COUNT(separacion_14d) sep_matured_day,
        SUM(minuta_60d) FILTER (WHERE minuta_60d IS NOT NULL) minuta_pos_day,
        COUNT(minuta_60d) minuta_matured_day
 FROM features.lead_evidence
 GROUP BY codigo_proyecto,decision_at::date
),
advisor_daily AS (
 SELECT asesor,decision_at::date d,
        COUNT(*) leads_day,
        SUM(separacion_14d) FILTER (WHERE separacion_14d IS NOT NULL) sep_pos_day,
        COUNT(separacion_14d) sep_matured_day,
        SUM(minuta_60d) FILTER (WHERE minuta_60d IS NOT NULL) minuta_pos_day,
        COUNT(minuta_60d) minuta_matured_day
 FROM features.lead_evidence
 WHERE asesor IS NOT NULL
 GROUP BY asesor,decision_at::date
),
global_daily AS (
 SELECT decision_at::date d,
        SUM(separacion_14d) FILTER (WHERE separacion_14d IS NOT NULL) sep_pos_day,
        COUNT(separacion_14d) sep_matured_day,
        SUM(minuta_60d) FILTER (WHERE minuta_60d IS NOT NULL) minuta_pos_day,
        COUNT(minuta_60d) minuta_matured_day
 FROM features.lead_evidence
 GROUP BY decision_at::date
)
SELECT
 e.evidence_key,e.decision_at,e.documento_cliente,e.codigo_proyecto,e.asesor,
 COALESCE((SELECT COUNT(*) FROM features.lead_evidence p
   WHERE p.documento_cliente=e.documento_cliente
     AND p.decision_at<e.decision_at
     AND p.decision_at>=e.decision_at-interval '90 days'),0) AS client_prior_assignments_90d,
 (SELECT EXTRACT(EPOCH FROM (e.decision_at-MAX(p.decision_at)))/86400.0
   FROM features.lead_evidence p
   WHERE p.documento_cliente=e.documento_cliente
     AND p.decision_at<e.decision_at) AS days_since_previous_assignment,
 COALESCE((SELECT SUM(pd.leads_day) FROM project_daily pd
   WHERE pd.codigo_proyecto=e.codigo_proyecto
     AND pd.d>=e.decision_at::date-90
     AND pd.d<e.decision_at::date),0) AS project_leads_90d,
 (SELECT SUM(pd.sep_pos_day)::double precision/NULLIF(SUM(pd.sep_matured_day),0)
   FROM project_daily pd
   WHERE pd.codigo_proyecto=e.codigo_proyecto
     AND pd.d>=e.decision_at::date-90
     AND pd.d<e.decision_at::date
     AND pd.d+{SEP_H}<=e.decision_at::date) AS project_sep_rate_90d,
 (SELECT SUM(pd.minuta_pos_day)::double precision/NULLIF(SUM(pd.minuta_matured_day),0)
   FROM project_daily pd
   WHERE pd.codigo_proyecto=e.codigo_proyecto
     AND pd.d>=e.decision_at::date-180
     AND pd.d<e.decision_at::date
     AND pd.d+{MINUTA_H}<=e.decision_at::date) AS project_minuta_rate_180d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE COALESCE((SELECT SUM(ad.leads_day)
   FROM advisor_daily ad
   WHERE ad.asesor=e.asesor
     AND ad.d>=e.decision_at::date-90
     AND ad.d<e.decision_at::date),0) END AS advisor_leads_90d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE (SELECT SUM(ad.sep_pos_day)::double precision/NULLIF(SUM(ad.sep_matured_day),0)
   FROM advisor_daily ad
   WHERE ad.asesor=e.asesor
     AND ad.d>=e.decision_at::date-90
     AND ad.d<e.decision_at::date
     AND ad.d+{SEP_H}<=e.decision_at::date) END AS advisor_sep_rate_90d,
 CASE WHEN e.asesor IS NULL THEN NULL ELSE (SELECT SUM(ad.minuta_pos_day)::double precision/NULLIF(SUM(ad.minuta_matured_day),0)
   FROM advisor_daily ad
   WHERE ad.asesor=e.asesor
     AND ad.d>=e.decision_at::date-180
     AND ad.d<e.decision_at::date
     AND ad.d+{MINUTA_H}<=e.decision_at::date) END AS advisor_minuta_rate_180d,
 (SELECT SUM(gd.sep_pos_day)::double precision/NULLIF(SUM(gd.sep_matured_day),0)
   FROM global_daily gd
   WHERE gd.d>=e.decision_at::date-90
     AND gd.d<e.decision_at::date
     AND gd.d+{SEP_H}<=e.decision_at::date) AS global_sep_rate_90d,
 (SELECT SUM(gd.minuta_pos_day)::double precision/NULLIF(SUM(gd.minuta_matured_day),0)
   FROM global_daily gd
   WHERE gd.d>=e.decision_at::date-180
     AND gd.d<e.decision_at::date
     AND gd.d+{MINUTA_H}<=e.decision_at::date) AS global_minuta_rate_180d
FROM target e
ORDER BY e.decision_at,e.evidence_key
"""


## 5. Benchmark 100 / 500 / 1000


In [6]:
BENCH_SIZES=[100,500,1000]
rows=[]
for n in BENCH_SIZES:
    for name,fn in [("current_correlated",baseline_query),("incremental_daily",incremental_query)]:
        t0=time.perf_counter()
        try:
            out=df(fn(n)); sec=time.perf_counter()-t0
            rows.append({"n":n,"method":name,"seconds":sec,"rows":len(out),"rows_per_second":len(out)/sec if sec else np.nan})
            print(f"{name} n={n}: {sec:.3f}s")
        except Exception as exc:
            conn.rollback()
            rows.append({"n":n,"method":name,"seconds":np.nan,"rows":0,"error":repr(exc)})
benchmark=pd.DataFrame(rows)
benchmark


C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


current_correlated n=100: 4.713s


C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


incremental_daily n=100: 3.408s


C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


...

C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


current_correlated n=1000: 87.151s


C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


incremental_daily n=1000: 27.449s


,n,method,seconds,rows,rows_per_second
0,100,current_correlated,4.712858,100,21.218547
1,100,incremental_daily,3.408203,100,29.340974
2,500,current_correlated,35.073829,500,14.255644
3,500,incremental_daily,11.915763,500,41.961224
4,1000,current_correlated,87.150777,1000,11.474367
5,1000,incremental_daily,27.449474,1000,36.430571


## 6. Speedup


In [7]:
pivot=benchmark.pivot(index="n",columns="method",values="seconds").reset_index()
if {"current_correlated","incremental_daily"}.issubset(pivot.columns):
    pivot["speedup_x"]=pivot["current_correlated"]/pivot["incremental_daily"]
pivot


method,n,current_correlated,incremental_daily,speedup_x
0,100,4.712858,3.408203,1.382798
1,500,35.073829,11.915763,2.943482
2,1000,87.150777,27.449474,3.174953


## 7. Equivalencia point-in-time


In [8]:
N_CHECK=100
base=df(baseline_query(N_CHECK))
inc=df(incremental_query(N_CHECK))

features=[
"client_prior_assignments_90d","days_since_previous_assignment",
"project_leads_90d","project_sep_rate_90d","project_minuta_rate_180d",
"advisor_leads_90d","advisor_sep_rate_90d","advisor_minuta_rate_180d",
"global_sep_rate_90d","global_minuta_rate_180d"
]

m=base.merge(inc,on="evidence_key",suffixes=("_base","_inc"))
checks=[]
for c in features:
    a=pd.to_numeric(m[f"{c}_base"],errors="coerce")
    b=pd.to_numeric(m[f"{c}_inc"],errors="coerce")
    ok=np.isclose(a.fillna(-999999),b.fillna(-999999),rtol=1e-9,atol=1e-9)
    checks.append({"feature":c,"match_pct":ok.mean(),"rows":len(ok),"max_abs_diff":float((a-b).abs().max()) if len(a) else np.nan})
equivalence=pd.DataFrame(checks)
equivalence


C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)
C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


,feature,match_pct,rows,max_abs_diff
0,client_prior_assignments_90d,1.00,100,0.000000
1,days_since_previous_assignment,1.00,100,0.000000
2,project_leads_90d,0.12,100,16.000000
3,project_sep_rate_90d,0.03,100,0.000132
4,project_minuta_rate_180d,0.02,100,0.000106
5,advisor_leads_90d,1.00,100,NaN
6,advisor_sep_rate_90d,1.00,100,NaN
7,advisor_minuta_rate_180d,1.00,100,NaN
8,global_sep_rate_90d,0.00,100,0.000094
9,global_minuta_rate_180d,0.00,100,0.000053


## 8. Diferencias concretas


In [9]:
diffs=[]
for c in features:
    a=pd.to_numeric(m[f"{c}_base"],errors="coerce")
    b=pd.to_numeric(m[f"{c}_inc"],errors="coerce")
    ok=np.isclose(a.fillna(-999999),b.fillna(-999999),rtol=1e-9,atol=1e-9)
    bad=m.loc[~ok,["evidence_key",f"{c}_base",f"{c}_inc"]].head(10).copy()
    if len(bad):
        bad["feature"]=c
        diffs.append(bad)
if diffs:
    diff_examples=pd.concat(diffs,ignore_index=True)
    display(diff_examples.head(100))
else:
    diff_examples=pd.DataFrame()
    print("Sin diferencias.")


,evidence_key,project_leads_90d_base,project_leads_90d_inc,feature,project_sep_rate_90d_base,project_sep_rate_90d_inc,project_minuta_rate_180d_base,project_minuta_rate_180d_inc,global_sep_rate_90d_base,global_sep_rate_90d_inc,global_minuta_rate_180d_base,global_minuta_rate_180d_inc
0,0ac5b3639c2552a683d6e1759d6cf4cd,1134.0,1133.0,project_leads_90d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,fe6c085e73a472d83e8c1d68eb6dc021,1886.0,1885.0,project_leads_90d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0a33059b275ab6d868ee86eb602226a4,1887.0,1885.0,project_leads_90d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,f4b2dac14536ce8c31ae3c6d7f520d7d,1134.0,1133.0,project_leads_90d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,e89eaadf6ea6f9846d8b17d4b87f3505,1134.0,1133.0,project_leads_90d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,d50bcd3e6e35d89d765d72dd5a880705,1352.0,1358.0,project_leads_90d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,eba9be30db56bf3405c08b8fa5a46750,2331.0,2333.0,project_leads_90d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,c44ad1ad52d5a66cfb26f5b0f3712bf2,2331.0,2333.0,project_leads_90d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,15e5623fab4e65839ae4eacf235840eb,2331.0,2333.0,project_leads_90d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,e58b4ee7843992671532d6baa37af9a2,1203.0,1207.0,project_leads_90d,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 9. EXPLAIN comparativo


In [10]:
for name,q in [("CURRENT",baseline_query(100)),("INCREMENTAL_DAILY",incremental_query(100))]:
    print("\n###",name)
    p=df("EXPLAIN "+q)
    print("\n".join(p.iloc[:,0].astype(str)))



### CURRENT
Subquery Scan on e  (cost=2.41..470952.74 rows=100 width=194)
  ->  Limit  (cost=2.41..204.34 rows=100 width=380)
        ->  Incremental Sort  (cost=2.41..4578.21 rows=2266 width=380)
              Sort Key: lead_evidence.decision_at, lead_evidence.evidence_key
              Presorted Key: lead_evidence.decision_at
              ->  Index Scan Backward using ix_lead_evidence_decision_at on lead_evidence  (cost=0.42..4476.27 rows=2266 width=380)
                    Index Cond: (decision_at >= (CURRENT_DATE - '14 days'::interval))
                    Filter: (features_refreshed_at IS NULL)
  SubPlan 1
    ->  Aggregate  (cost=8.45..8.46 rows=1 width=8)
          ->  Index Only Scan using ix_lead_evidence_document_time on lead_evidence p  (cost=0.42..8.44 rows=1 width=0)
                Index Cond: ((documento_cliente = e.documento_cliente) AND (decision_at < e.decision_at) AND (decision_at >= (e.decision_at - '90 days'::interval)))
  SubPlan 3
    ->  Result  (cost=8.44..8.

C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)
C:\Users\dinat\AppData\Local\Temp\ipykernel_9800\2874589508.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql,conn,params=params)


## 10. EXPLAIN ANALYZE opcional


In [11]:
RUN_EXPLAIN_ANALYZE=False
if RUN_EXPLAIN_ANALYZE:
    for name,q in [("CURRENT",baseline_query(100)),("INCREMENTAL_DAILY",incremental_query(100))]:
        print("\n###",name)
        p=df("EXPLAIN (ANALYZE,BUFFERS) "+q)
        print("\n".join(p.iloc[:,0].astype(str)))
else:
    print("RUN_EXPLAIN_ANALYZE=False")


RUN_EXPLAIN_ANALYZE=False


## 11. Estimación runtime LIVE


In [12]:
pending_live=int(status.iloc[0]["pending_live"])
est=[]
for method in benchmark["method"].unique():
    x=benchmark[(benchmark["method"]==method)&benchmark["rows_per_second"].notna()&benchmark["rows_per_second"].gt(0)]
    if len(x):
        r=x.sort_values("n").iloc[-1]
        rps=float(r["rows_per_second"])
        sec=pending_live/rps
        est.append({"method":method,"reference_rps":rps,"estimated_seconds_live":sec,"estimated_minutes_live":sec/60})
live_estimates=pd.DataFrame(est)
live_estimates


,method,reference_rps,estimated_seconds_live,estimated_minutes_live
0,current_correlated,11.474367,175.347363,2.922456
1,incremental_daily,36.430571,55.228342,0.920472


## 12. Full LIVE opcional


In [13]:
RUN_FULL_LIVE=False
if RUN_FULL_LIVE:
    n=int(status.iloc[0]["pending_live"])
    rows=[]
    for name,fn in [("current_correlated",baseline_query),("incremental_daily",incremental_query)]:
        t0=time.perf_counter()
        out=df(fn(n))
        sec=time.perf_counter()-t0
        rows.append({"method":name,"rows":len(out),"seconds":sec,"minutes":sec/60})
    full_live=pd.DataFrame(rows)
    display(full_live)
else:
    print("RUN_FULL_LIVE=False")


RUN_FULL_LIVE=False


## 13. Diseño persistente recomendado

Si `incremental_daily` gana y mantiene equivalencia:

```text
features.project_history_daily
features.advisor_history_daily
features.global_history_daily
```

con índices:

```text
(codigo_proyecto, fecha)
(asesor, fecha)
(fecha)
```

El cliente puede seguir resolviéndose directamente con `(documento_cliente, decision_at)` porque su historial es mucho más selectivo.


## 14. Gates


In [14]:
gates=[]
def add(name,passed,detail):
    gates.append({"gate":name,"status":"PASS" if passed else "FAIL","detail":detail})

mm=float(equivalence["match_pct"].min()) if len(equivalence) else np.nan
add("Equivalencia point-in-time",pd.notna(mm) and mm>=0.999,f"min_match={mm:.3%}" if pd.notna(mm) else "N/A")

pv=benchmark.pivot(index="n",columns="method",values="seconds").dropna()
if len(pv) and {"current_correlated","incremental_daily"}.issubset(pv.columns):
    r=pv.iloc[-1]
    sx=float(r["current_correlated"]/r["incremental_daily"])
    add("Speedup material",sx>=1.5,f"speedup={sx:.2f}x")

add("LIVE incremental",int(status.iloc[0]["pending_live"])>0,f"pending_live={int(status.iloc[0]['pending_live']):,}")

gate_table=pd.DataFrame(gates)
gate_table


,gate,status,detail
0,Equivalencia point-in-time,FAIL,min_match=0.000%
1,Speedup material,PASS,speedup=3.17x
2,LIVE incremental,PASS,"pending_live=2,012"


## 15. Smart insights


In [15]:
print("=== 01E LEAD FEATURES INCREMENTAL LAB ===")
print(f"1. Pendientes históricos: {int(status.iloc[0]['pending']):,}")
print(f"2. Pendientes live: {int(status.iloc[0]['pending_live']):,}")
if len(equivalence):
    print(f"3. Equivalencia mínima: {equivalence['match_pct'].min():.1%}")
if len(pv) and {"current_correlated","incremental_daily"}.issubset(pv.columns):
    r=pv.iloc[-1]
    print(f"4. Speedup mayor benchmark: {float(r['current_correlated']/r['incremental_daily']):.2f}x")
display(live_estimates)


=== 01E LEAD FEATURES INCREMENTAL LAB ===
1. Pendientes históricos: 206,029
2. Pendientes live: 2,012
3. Equivalencia mínima: 0.0%
4. Speedup mayor benchmark: 3.17x


,method,reference_rps,estimated_seconds_live,estimated_minutes_live
0,current_correlated,11.474367,175.347363,2.922456
1,incremental_daily,36.430571,55.228342,0.920472


## 16. Cambios deshabilitados


In [16]:
CREATE_PERSISTENT_TABLES=False
APPLY_INDEXES=False
APPLY_INCREMENTAL_FEATURES=False

print("CREATE_PERSISTENT_TABLES =",CREATE_PERSISTENT_TABLES)
print("APPLY_INDEXES =",APPLY_INDEXES)
print("APPLY_INCREMENTAL_FEATURES =",APPLY_INCREMENTAL_FEATURES)


CREATE_PERSISTENT_TABLES = False
APPLY_INDEXES = False
APPLY_INCREMENTAL_FEATURES = False


In [17]:
conn.close(); print("Conexión cerrada.")


Conexión cerrada.
